# RE-RL: Генерация данных из формальных доказательств Lean 4

Этот notebook демонстрирует генерацию training data для обучения theorem provers.

**Подход**: Реализация алгоритма из статьи **LeanNavigator** — систематическое исследование графов переходов состояний.

## Что мы делаем:
1. Берём теорему из Lean репозитория
2. Исследуем граф состояний через BFS
3. Собираем пары `(state, tactic)` для обучения LLM

## 1. Установка зависимостей

Для работы нужны:
- `lean-dojo` — Python API для Lean 4
- `elan` — Lean version manager (устанавливается отдельно)

In [ ]:
# Установка lean-dojo (если ещё не установлен)
# !pip install lean-dojo

# Установка elan (Lean version manager) - выполнить в терминале:
# curl -sSf https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh

In [ ]:
# Проверяем установку
import lean_dojo
print(f"LeanDojo version: {lean_dojo.__version__}")

## 2. Импорты

In [ ]:
from lean_dojo import LeanGitRepo, Theorem, Dojo, trace
import time
import json

# RE-RL formal math модуль
from re_rl.tasks.formal import (
    StateExplorer,
    TacticGenerator,
    generate_tactics,
    ExplorationResult,
)

## 3. Подготовка репозитория

Используем простой пример `lean4-example`. 

**Важно**: При первом запуске LeanDojo делает "tracing" репозитория (компиляция + анализ). Это занимает время, но результат кэшируется.

In [ ]:
# Репозиторий с примерами теорем
repo = LeanGitRepo(
    "https://github.com/yangky11/lean4-example",
    "7b6ecb9ad4829e4e73600a3329baeb3b5df8d23f"
)

print(f"Репозиторий: {repo}")

In [ ]:
# Tracing репозитория (один раз, потом кэшируется)
print("Tracing репозитория (может занять ~1 минуту при первом запуске)...")
start = time.time()

traced_repo = trace(repo)

print(f"✓ Tracing завершён за {time.time() - start:.1f}с")
print(f"Файлов в репозитории: {len(traced_repo.traced_files)}")

## 4. Выбор теоремы

In [ ]:
# Теорема: a + b + c = a + c + b
theorem = Theorem(repo, "Lean4Example.lean", "hello_world")

print(f"Теорема: {theorem.full_name}")
print(f"Файл: {theorem.file_path}")

## 5. Базовый пример: применение одной тактики

In [ ]:
# Простой пример работы с Dojo
with Dojo(theorem) as (dojo, state_0):
    print("Начальное состояние:")
    print("-" * 40)
    print(state_0.pp)
    print("-" * 40)
    
    # Применяем тактику
    result = dojo.run_tac(state_0, "omega")
    
    print(f"\nПосле 'omega': {type(result).__name__}")
    if type(result).__name__ == "ProofFinished":
        print("✓ Доказательство завершено!")

## 6. Генерация тактик (rule-based)

Наш генератор создаёт кандидаты тактик на основе:
- Базовых тактик (`rfl`, `ring`, `simp`, `omega`, ...)
- Шаблонов с гипотезами (`apply {h}`, `rw [{h}]`, ...)
- Структурных тактик (`cases`, `induction`, ...)

In [ ]:
# Генерируем тактики для состояния
state_pp = "a b c : Nat\n⊢ a + b + c = a + c + b"

tactics = generate_tactics(state_pp, max_count=50)

print(f"Сгенерировано {len(tactics)} тактик:")
for i, tactic in enumerate(tactics[:20]):
    print(f"  {i+1}. {tactic}")
print("  ...")

## 7. Полный BFS как в LeanNavigator

Теперь запустим полноценное исследование графа состояний.

In [ ]:
# Создаём StateExplorer с параметрами
explorer = StateExplorer(
    max_steps=500,      # Макс. применений тактик
    max_time=60,        # Макс. время (секунды)
    max_depth=8,        # Макс. глубина поиска
    max_states=100,     # Макс. уникальных состояний
    verbose=False,
)

print("Настройки explorer:")
print(f"  max_steps: {explorer.max_steps}")
print(f"  max_time: {explorer.max_time}s")
print(f"  max_depth: {explorer.max_depth}")

In [ ]:
# Запускаем BFS исследование
print("Запускаем BFS исследование графа состояний...")
print()

start = time.time()

with Dojo(theorem) as (dojo, state_0):
    print(f"Начальное состояние: {state_0.pp}")
    print()
    
    result, training_pairs, stats = explorer.explore(
        dojo,
        state_0,
        theorem_name=theorem.full_name,
        exit_on_proof=False,  # Продолжаем исследование после первого доказательства
    )

elapsed = time.time() - start

print("=" * 60)
print("РЕЗУЛЬТАТЫ ИССЛЕДОВАНИЯ")
print("=" * 60)
print(f"Результат: {result.value}")
print(f"Время: {elapsed:.2f}с")
print()
print(f"Статистика:")
print(f"  Всего состояний: {stats.total_states}")
print(f"  Тактик попробовано: {stats.total_tactics_tried}")
print(f"  Успешных тактик: {stats.successful_tactics}")
print(f"  Доказательство найдено: {stats.proof_found}")
print(f"  Максимальная глубина: {stats.max_depth_reached}")
print()
print(f"Training pairs: {len(training_pairs)}")

## 8. Анализ Training Pairs

In [ ]:
# Пары, ведущие к доказательству (distance_to_proof >= 0)
proof_pairs = [p for p in training_pairs if p.distance_to_proof >= 0]

print(f"Пары на пути к ProofFinished: {len(proof_pairs)}")
print()

for pair in proof_pairs:
    state_short = pair.state[:50].replace('\n', ' ')
    print(f"[distance={pair.distance_to_proof}] '{pair.tactic}'")
    print(f"  State: {state_short}...")
    print()

In [ ]:
# Смотрим все уникальные тактики
unique_tactics = set(p.tactic for p in training_pairs)

print(f"Уникальных тактик: {len(unique_tactics)}")
print()
for tactic in sorted(unique_tactics)[:20]:
    count = sum(1 for p in training_pairs if p.tactic == tactic)
    print(f"  {tactic}: {count} раз")

## 9. Сохранение датасета

In [ ]:
# Сохраняем в JSONL формате (для обучения)
output_file = "lean_training_data.jsonl"

with open(output_file, "w") as f:
    for pair in training_pairs:
        record = {
            "state": pair.state,
            "tactic": pair.tactic,
            "next_state": pair.next_state,
            "distance_to_proof": pair.distance_to_proof,
            "theorem_name": pair.theorem_name,
        }
        f.write(json.dumps(record) + "\n")

print(f"✓ Сохранено {len(training_pairs)} пар в {output_file}")

In [ ]:
# Смотрим результат
!head -5 lean_training_data.jsonl

## 10. SFT формат для обучения

Преобразуем в формат для Supervised Fine-Tuning.

In [ ]:
# Создаём SFT датасет
sft_data = []

for pair in training_pairs:
    sft_record = {
        "instruction": "You are a Lean 4 theorem prover. Given the current proof state, suggest the next tactic.",
        "input": f"State:\n{pair.state}",
        "output": pair.tactic,
    }
    sft_data.append(sft_record)

# Сохраняем
with open("lean_sft_data.json", "w") as f:
    json.dump(sft_data, f, indent=2)

print(f"✓ SFT датасет: {len(sft_data)} примеров")
print()
print("Пример:")
print(json.dumps(sft_data[0], indent=2))

## 11. Исследование нескольких теорем

Для реального датасета нужно исследовать много теорем.

In [ ]:
# Список теорем в репозитории
theorems = [
    Theorem(repo, "Lean4Example.lean", "hello_world"),
    Theorem(repo, "Lean4Example.lean", "foo"),
]

all_pairs = []

for thm in theorems:
    print(f"Исследуем: {thm.full_name}")
    
    try:
        with Dojo(thm) as (dojo, state_0):
            result, pairs, stats = explorer.explore(
                dojo, state_0, 
                theorem_name=thm.full_name,
                exit_on_proof=False,
            )
            
            all_pairs.extend(pairs)
            print(f"  ✓ {stats.total_states} состояний, {len(pairs)} пар, proof: {stats.proof_found}")
    except Exception as e:
        print(f"  ✗ Ошибка: {e}")

print()
print(f"Всего: {len(all_pairs)} training pairs")

## 12. Сравнение с LeanNavigator

| Компонент | LeanNavigator | Наш RE-RL |
|-----------|---------------|----------|
| LeanDojo взаимодействие | ✓ | ✓ |
| BFS по графу состояний | ✓ | ✓ |
| Приоритетная очередь | ✓ | ✓ |
| Rule-based тактики | ✓ | ✓ |
| Embedding retrieval (BERT+FAISS) | ✓ | ❌ |
| Ray параллелизация | 24 процесса | ❌ |

**Для масштабирования** нужно добавить:
1. Embedding retrieval для ускорения генерации тактик (~100x)
2. Параллелизацию через Ray
3. Использовать Mathlib4 (требует tracing ~часы)

## Заключение

Мы реализовали основной алгоритм LeanNavigator:

1. **BFS исследование** графа переходов состояний
2. **Генерация тактик** (rule-based)
3. **Сбор training pairs** `(state, tactic)`
4. **Экспорт** в форматы для обучения LLM

Это позволяет генерировать данные для обучения theorem provers на любых Lean 4 репозиториях.